In [1]:
%pip install --no-cache-dir \
    delta-spark==3.2.0 \
    minio \
    boto3 \
    psycopg2-binary \
    faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 879.5 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 5.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 3.5 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 2.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 1.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 1.7 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
# ==============================================================================
# 🏛️ LAKEHOUSE CONFIGURATION CENTER
# ==============================================================================
import os

# --- Versões do Ecossistema (Matriz de Compatibilidade Spark 3.5.x) ---
VERSIONS = {
    "SPARK": "3.5.0",
    "SCALA": "2.12",
    "DELTA": "3.2.0",
    "HADOOP_AWS": "3.3.4",
    "AWS_SDK": "1.12.262",
    "POSTGRES_JDBC": "42.6.0"
}

# --- Caminhos e Drivers ---
WORK_DIR = "/home/jovyan/work"
DATA_DIR = "/home/jovyan/data"
DRIVERS_DIR = os.path.join(DATA_DIR, "drivers")
POSTGRES_JAR_NAME = f"postgresql-{VERSIONS['POSTGRES_JDBC']}.jar"
POSTGRES_JAR_PATH = os.path.join(DRIVERS_DIR, POSTGRES_JAR_NAME)

# --- Credenciais (Em Prod, usaríamos Variáveis de Ambiente ou Secrets Manager) ---
CONFIG = {
    "MINIO": {
        "ENDPOINT": "http://minio:9000",
        "ACCESS_KEY": "minio_admin",
        "SECRET_KEY": "minio_password",
        "BUCKET_BRONZE": "bronze",
        "BUCKET_SILVER": "silver",
        "BUCKET_GOLD": "gold"
    },
    "POSTGRES": {
        "HOST": "lakehouse-postgres",
        "PORT": "5432",
        "DB": "real_estate_dw",
        "USER": "admin",
        "PASS": "admin_password",
        "DRIVER_CLASS": "org.postgresql.Driver",
        "URL": "jdbc:postgresql://lakehouse-postgres:5432/real_estate_dw"
    }
}

print("✅ Configurações carregadas.")

✅ Configurações carregadas.


In [3]:
import requests
import sys

def ensure_driver_exists(url, local_path):
    """
    Verifica se o driver existe. Se não, faz download.
    """
    if os.path.exists(local_path):
        print(f"📦 Driver já existe em: {local_path}")
        return True
    
    # Cria diretório se não existir
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    
    print(f"⬇️ Baixando driver de: {url} ...")
    try:
        response = requests.get(url, timeout=60)
        if response.status_code == 200:
            with open(local_path, 'wb') as f:
                f.write(response.content)
            print(f"✅ Download concluído: {local_path}")
            return True
        else:
            print(f"❌ Erro HTTP {response.status_code} ao baixar driver.")
            return False
    except Exception as e:
        print(f"❌ Falha de conexão: {e}")
        return False

# URL Oficial Maven
MAVEN_URL = f"https://repo1.maven.org/maven2/org/postgresql/postgresql/{VERSIONS['POSTGRES_JDBC']}/{POSTGRES_JAR_NAME}"

# Executa verificação
if not ensure_driver_exists(MAVEN_URL, POSTGRES_JAR_PATH):
    raise Exception("⛔ CRÍTICO: Não foi possível obter o driver JDBC. O Spark não pode iniciar.")

📦 Driver já existe em: /home/jovyan/data/drivers/postgresql-42.6.0.jar


In [4]:
from pyspark.sql import SparkSession
from delta import *

def create_lakehouse_session(app_name="Lakehouse Production"):
    
    maven_packages = [
        f"io.delta:delta-spark_2.12:{VERSIONS['DELTA']}",
        f"org.apache.hadoop:hadoop-aws:{VERSIONS['HADOOP_AWS']}",
        f"com.amazonaws:aws-java-sdk-bundle:{VERSIONS['AWS_SDK']}"
    ]
    
    conf = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .config("spark.driver.memory", "4g") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.jars", POSTGRES_JAR_PATH) \
        .config("spark.driver.extraClassPath", POSTGRES_JAR_PATH) \
        .config("spark.executor.extraClassPath", POSTGRES_JAR_PATH) \
        .config("spark.jars.packages", ",".join(maven_packages)) \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .config("spark.hadoop.fs.s3a.endpoint", CONFIG["MINIO"]["ENDPOINT"]) \
        .config("spark.hadoop.fs.s3a.access.key", CONFIG["MINIO"]["ACCESS_KEY"]) \
        .config("spark.hadoop.fs.s3a.secret.key", CONFIG["MINIO"]["SECRET_KEY"]) \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
        .config("spark.databricks.delta.retentionDurationCheck.enabled", "false")

    return conf.getOrCreate()

# Inicia a Sessão
print("⏳ Iniciando Spark Session (Bootstrapping)...")
spark = create_lakehouse_session()
spark.sparkContext.setLogLevel("ERROR")
print(f"🚀 Spark Session Ativa! Versão: {spark.version}")

⏳ Iniciando Spark Session (Bootstrapping)...
🚀 Spark Session Ativa! Versão: 3.5.0


In [5]:
def run_healthcheck():
    print("🏥 INICIANDO DIAGNÓSTICO DO LAKEHOUSE...\n")
    
    # 1. Teste de Driver JDBC (Classpath)
    print("1️⃣  Verificando Classpath do Java...")
    try:
        spark.sparkContext._gateway.jvm.java.lang.Class.forName(CONFIG["POSTGRES"]["DRIVER_CLASS"])
        print(f"   ✅ Driver {CONFIG['POSTGRES']['DRIVER_CLASS']} carregado.")
    except Exception as e:
        print(f"   ❌ ERRO DE CLASSPATH: {e}")
        return

    # 2. Teste de Conectividade PostgreSQL
    print("\n2️⃣  Testando Conexão PostgreSQL...")
    try:
        # Tenta ler apenas 1 linha da view interna de tabelas para ser rápido
        df_pg = spark.read \
            .format("jdbc") \
            .option("url", CONFIG["POSTGRES"]["URL"]) \
            .option("dbtable", "information_schema.tables") \
            .option("user", CONFIG["POSTGRES"]["USER"]) \
            .option("password", CONFIG["POSTGRES"]["PASS"]) \
            .option("driver", CONFIG["POSTGRES"]["DRIVER_CLASS"]) \
            .load() \
            .limit(1)
        df_pg.collect()
        print("   ✅ Conexão JDBC estabelecida com sucesso.")
    except Exception as e:
        print(f"   ❌ FALHA NO POSTGRES: {e}")

    # 3. Teste de Escrita e Leitura no MinIO (Delta Lake)
    print("\n3️⃣  Testando Delta Lake no MinIO...")
    try:
        test_path = f"s3a://{CONFIG['MINIO']['BUCKET_BRONZE']}/_healthcheck"
        spark.range(5).write.format("delta").mode("overwrite").save(test_path)
        count = spark.read.format("delta").load(test_path).count()
        if count == 5:
            print(f"   ✅ Escrita e Leitura Delta OK (Path: {test_path})")
        else:
            print("   ⚠️  Alerta: Contagem de linhas divergiu.")
    except Exception as e:
        print(f"   ❌ FALHA NO MINIO/DELTA: {e}")

    print("\n🏁 DIAGNÓSTICO CONCLUÍDO.")

# Rodar Diagnóstico
run_healthcheck()

🏥 INICIANDO DIAGNÓSTICO DO LAKEHOUSE...

1️⃣  Verificando Classpath do Java...
   ✅ Driver org.postgresql.Driver carregado.

2️⃣  Testando Conexão PostgreSQL...
   ✅ Conexão JDBC estabelecida com sucesso.

3️⃣  Testando Delta Lake no MinIO...
   ✅ Escrita e Leitura Delta OK (Path: s3a://bronze/_healthcheck)

🏁 DIAGNÓSTICO CONCLUÍDO.


In [7]:
from pyspark.sql.functions import current_timestamp, lit

def ingest_table_to_bronze(table_name, target_bucket):
    """
    Lê do Postgres e salva no S3 (Bronze) em formato Delta
    """
    print(f"🔄 Ingerindo tabela '{table_name}' para {target_bucket}...")
    
    # 1. Leitura (Source)
    df_source = spark.read \
        .format("jdbc") \
        .option("url", CONFIG["POSTGRES"]["URL"]) \
        .option("dbtable", f"public.{table_name}") \
        .option("user", CONFIG["POSTGRES"]["USER"]) \
        .option("password", CONFIG["POSTGRES"]["PASS"]) \
        .option("driver", CONFIG["POSTGRES"]["DRIVER_CLASS"]) \
        .load()
    
    # 2. Enriquecimento (Metadata)
    df_enriched = df_source.withColumn("ingestion_date", current_timestamp())
    
    # 3. Escrita (Sink)
    target_path = f"s3a://{target_bucket}/{table_name}"
    
    df_enriched.write \
        .format("delta") \
        .mode("overwrite") \
        .save(target_path)
    
    print(f"✨ Sucesso! Dados salvos em: {target_path}")
    return spark.read.format("delta").load(target_path)

# --- Criando dados falsos no Postgres para testar a ingestão ---
# (Apenas para demonstração, na vida real a tabela já existiria)
fake_data = spark.createDataFrame([(1, "Apartamento Luxo", 1500000), (2, "Casa Simples", 300000)], ["id", "imovel", "valor"])
fake_data.write \
    .format("jdbc") \
    .option("url", CONFIG["POSTGRES"]["URL"]) \
    .option("dbtable", "public.imoveis_venda") \
    .option("user", CONFIG["POSTGRES"]["USER"]) \
    .option("password", CONFIG["POSTGRES"]["PASS"]) \
    .option("driver", CONFIG["POSTGRES"]["DRIVER_CLASS"]) \
    .mode("overwrite") \
    .save()

# --- Executando a Pipeline de Ingestão ---
df_bronze = ingest_table_to_bronze("imoveis_venda", CONFIG["MINIO"]["BUCKET_BRONZE"])
df_bronze.show()

🔄 Ingerindo tabela 'imoveis_venda' para bronze...
✨ Sucesso! Dados salvos em: s3a://bronze/imoveis_venda
+---+----------------+-------+--------------------+
| id|          imovel|  valor|      ingestion_date|
+---+----------------+-------+--------------------+
|  2|    Casa Simples| 300000|2026-03-02 23:52:...|
|  1|Apartamento Luxo|1500000|2026-03-02 23:52:...|
+---+----------------+-------+--------------------+



In [8]:
import boto3
from botocore.exceptions import ClientError

def init_data_lake_buckets():
    """
    Verifica e cria os buckets (Bronze, Silver, Gold) no MinIO.
    Garante que o Storage esteja pronto antes do Spark tentar escrever.
    """
    print("🏗️  INICIALIZANDO INFRAESTRUTURA DE STORAGE (MinIO)...")
    
    # Configuração do Cliente S3 (Boto3)
    s3_client = boto3.client(
        "s3",
        endpoint_url=CONFIG["MINIO"]["ENDPOINT"],
        aws_access_key_id=CONFIG["MINIO"]["ACCESS_KEY"],
        aws_secret_access_key=CONFIG["MINIO"]["SECRET_KEY"]
    )

    buckets_to_create = [
        CONFIG["MINIO"]["BUCKET_BRONZE"],
        CONFIG["MINIO"]["BUCKET_SILVER"],
        CONFIG["MINIO"]["BUCKET_GOLD"]
    ]

    for bucket_name in buckets_to_create:
        try:
            s3_client.create_bucket(Bucket=bucket_name)
            print(f"   ✅ Bucket criado: {bucket_name}")
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code in ('BucketAlreadyOwnedByYou', 'BucketAlreadyExists'):
                print(f"   ℹ️  Bucket já existe: {bucket_name}")
            else:
                print(f"   ❌ Erro ao criar bucket {bucket_name}: {e}")
                raise e
    
    print("✅ Infraestrutura de Storage Pronta.\n")

# Executa a inicialização
init_data_lake_buckets()

🏗️  INICIALIZANDO INFRAESTRUTURA DE STORAGE (MinIO)...
   ℹ️  Bucket já existe: bronze
   ℹ️  Bucket já existe: silver
   ℹ️  Bucket já existe: gold
✅ Infraestrutura de Storage Pronta.



In [11]:
spark.version

'3.5.0'

In [12]:
spark.sparkContext.getConf().getAll()

[('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false'),
 ('spark.hadoop.fs.s3a.path.style.access', 'true'),
 ('spark.driver.memory', '4g'),
 ('spark.files',
  'file:///home/jovyan/.ivy2/jars/io.delta_delta-spark_2.12-3.2.0.jar,file:///home/jovyan/.ivy2/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar,file:///home/jovyan/.ivy2/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar,file:///home/jovyan/.ivy2/jars/io.delta_delta-storage-3.2.0.jar,file:///home/jovyan/.ivy2/jars/org.antlr_antlr4-runtime-4.9.3.jar,file:///home/jovyan/.ivy2/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar'),
 ('spark.app.submitTime', '1772495545216'),
 ('spark.driver.extraClassPath',
  '/home/jovyan/data/drivers/postgresql-42.6.0.jar'),
 ('spark.submit.pyFiles',
  '/home/jovyan/.ivy2/jars/io.delta_delta-spark_2.12-3.2.0.jar,/home/jovyan/.ivy2/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar,/home/jovyan/.ivy2/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar,/home/jovyan/.ivy2/jars/io.delta_delta-storage-3.2.0.

In [13]:
for k, v in sorted(spark.sparkContext.getConf().getAll()):
    print(f"{k} = {v}")

spark.app.id = local-1772495546518
spark.app.initial.file.urls = file:///home/jovyan/.ivy2/jars/io.delta_delta-spark_2.12-3.2.0.jar,file:///home/jovyan/.ivy2/jars/org.apache.hadoop_hadoop-aws-3.3.4.jar,file:///home/jovyan/.ivy2/jars/io.delta_delta-storage-3.2.0.jar,file:///home/jovyan/.ivy2/jars/com.amazonaws_aws-java-sdk-bundle-1.12.262.jar,file:///home/jovyan/.ivy2/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar,file:///home/jovyan/.ivy2/jars/org.antlr_antlr4-runtime-4.9.3.jar
spark.app.initial.jar.urls = spark://0704a7d9ef91:36687/jars/postgresql-42.6.0.jar
spark.app.name = Lakehouse Production
spark.app.startTime = 1772495545498
spark.app.submitTime = 1772495545216
spark.databricks.delta.retentionDurationCheck.enabled = false
spark.driver.extraClassPath = /home/jovyan/data/drivers/postgresql-42.6.0.jar
spark.driver.extraJavaOptions = -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.

In [14]:
spark.sparkContext.getConf().get("spark.jars")

'/home/jovyan/data/drivers/postgresql-42.6.0.jar'

In [15]:
spark.sparkContext.getConf().get("spark.jars.packages")

'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262'

In [16]:
spark.conf.get("spark.sql.extensions")

'io.delta.sql.DeltaSparkSessionExtension'

In [17]:
spark.conf.get("spark.sql.catalog.spark_catalog")

'org.apache.spark.sql.delta.catalog.DeltaCatalog'

In [18]:
spark.conf.get("spark.hadoop.fs.s3a.endpoint")
spark.conf.get("spark.hadoop.fs.s3a.access.key")
spark.conf.get("spark.hadoop.fs.s3a.path.style.access")

'true'

In [19]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [20]:
spark.sql("SET spark.sql.extensions").show(truncate=False)

+--------------------+---------------------------------------+
|key                 |value                                  |
+--------------------+---------------------------------------+
|spark.sql.extensions|io.delta.sql.DeltaSparkSessionExtension|
+--------------------+---------------------------------------+



In [21]:
spark.sparkContext.getConf().get("spark.driver.memory")

'4g'

In [22]:
spark.sparkContext.master

'local[*]'

In [23]:
spark.sparkContext.getConf().get("spark.driver.extraClassPath")

'/home/jovyan/data/drivers/postgresql-42.6.0.jar'

In [25]:
df = spark.range(50_000_000)
df.cache()
df.count()
df.count()

50000000

In [27]:
df2 = df.filter("id > 5")

In [28]:
df2.count()

4

In [29]:
df = spark.range(1_000_000)
df2 = df.withColumn("x", df.id * 2)
df3 = df2.groupBy().sum("x")
df3.show()

+------------+
|      sum(x)|
+------------+
|999999000000|
+------------+



In [30]:
df.rdd.getNumPartitions()

12

In [31]:
df = df.repartition(8)
df.rdd.getNumPartitions()

8

In [32]:
df.write.mode("overwrite").parquet("s3a://bronze/test")

In [33]:
df.write.format("delta").mode("overwrite").save("s3a://silver/test_delta")

In [35]:
tracker = spark.sparkContext.statusTracker()

job_ids = tracker.getActiveJobIds()
job_ids

AttributeError: 'StatusTracker' object has no attribute 'getActiveJobIds'

In [41]:
from faker import Faker
import random

fake = Faker()

data = []

for _ in range(100000):
    data.append((
        fake.date_this_year().isoformat(),
        random.choice(["Eletronicos", "Roupas", "Casa"]),
        random.randint(10, 1000)
    ))

df = spark.createDataFrame(data, ["data_venda", "categoria", "valor"])
df.show(5)

+----------+-----------+-----+
|data_venda|  categoria|valor|
+----------+-----------+-----+
|2026-01-01|       Casa|  683|
|2026-02-18|Eletronicos|  782|
|2026-01-30|     Roupas|   16|
|2026-01-11|Eletronicos|  504|
|2026-02-01|Eletronicos|  209|
+----------+-----------+-----+
only showing top 5 rows



In [43]:
df.write.mode("overwrite").parquet("s3a://landing/vendas_raw")

In [44]:
spark.sparkContext._jsc.hadoopConfiguration().get("fs.s3a.endpoint")

'http://minio:9000'

In [45]:
df.write.mode("overwrite").parquet("s3a://landing/teste")

In [46]:
spark.read.parquet("s3a://landing/teste").show()

+----------+-----------+-----+
|data_venda|  categoria|valor|
+----------+-----------+-----+
|2026-02-09|     Roupas|  514|
|2026-01-25|     Roupas|  430|
|2026-02-08|Eletronicos|  431|
|2026-02-28|       Casa|  852|
|2026-01-07|     Roupas|  684|
|2026-01-03|     Roupas|  360|
|2026-01-28|       Casa|  445|
|2026-01-26|Eletronicos|  397|
|2026-02-28|Eletronicos|  109|
|2026-02-18|       Casa|  833|
|2026-02-17|     Roupas|  365|
|2026-02-18|Eletronicos|  888|
|2026-01-22|Eletronicos|  768|
|2026-01-31|     Roupas|  817|
|2026-02-06|       Casa|  130|
|2026-01-26|       Casa|  807|
|2026-01-16|       Casa|  559|
|2026-02-12|     Roupas|  196|
|2026-02-07|     Roupas|  113|
|2026-01-23|     Roupas|  302|
+----------+-----------+-----+
only showing top 20 rows



In [47]:
from pyspark.sql.functions import when

df_ml = df_silver.withColumn(
    "label",
    when(df_silver.valor > 500, 1).otherwise(0)
)

NameError: name 'df_silver' is not defined